In [1]:
import requests

In [2]:
standings_url = "https://fbref.com/en/comps/9/Premier-League-Stats"
standings_url2 = "https://fbref.com/en/comps/11/Serie-A-Stats"

In [3]:
data = requests.get(standings_url2)

<Response [429]>

In [4]:
from bs4 import BeautifulSoup
import time

In [5]:
soup = BeautifulSoup(data.text)
time.sleep(5)
soup
standings_table = soup.select('table.stats_table')[0]
links = standings_table.find_all('a')
links = [l.get("href") for l in links]
links = [l for l in links if '/squads/' in l]

IndexError: list index out of range

In [51]:
team_urls = [f"https://fbref.com{l}" for l in links]

In [52]:
data = requests.get(team_urls[0])

In [53]:
import pandas as pd
matches = pd.read_html(data.text, match="Scores & Fixtures")[0]

/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykernel_9861/4209044294.py:2: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]


ValueError: No tables found

In [32]:
soup = BeautifulSoup(data.text)
links = soup.find_all('a')
links = [l.get("href") for l in links]
links = [l for l in links if l and 'all_comps/shooting/' in l]

In [33]:
data = requests.get(f"https://fbref.com{links[0]}")

In [34]:
shooting = pd.read_html(data.text, match="Shooting")[0]

/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykernel_9861/2517714723.py:1: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  shooting = pd.read_html(data.text, match="Shooting")[0]


In [37]:
shooting.head()

,Date,Time,Comp,Round,Day,Venue,Result,GF,GA,Opponent,...,Dist,FK,PK,PKatt,xG,npxG,npxG/Sh,G-xG,np:G-xG,Match Report
0,2024-08-10,21:15,Coppa Italia,First round,Sat,Home,D,0 (4),0 (3),Modena,...,NaN,NaN,0,0,NaN,NaN,NaN,NaN,NaN,Match Report
1,2024-08-18,18:30,Serie A,Matchweek 1,Sun,Away,L,0,3,Hellas Verona,...,17.8,0.0,0,0,1.0,1.0,0.07,-1.0,-1.0,Match Report
2,2024-08-25,20:45,Serie A,Matchweek 2,Sun,Home,W,3,0,Bologna,...,17.5,0.0,0,0,2.5,2.5,0.15,0.5,0.5,Match Report
3,2024-08-31,20:45,Serie A,Matchweek 3,Sat,Home,W,2,1,Parma,...,17.5,1.0,0,0,2.2,2.2,0.08,-0.2,-0.2,Match Report
4,2024-09-15,18:00,Serie A,Matchweek 4,Sun,Away,W,4,0,Cagliari,...,16.0,0.0,0,0,1.9,1.9,0.15,2.1,2.1,Match Report


In [36]:
shooting.columns = shooting.columns.droplevel()

In [38]:
team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")

In [40]:
years = list(range(2024, 2022, -1))
all_matches = []

In [41]:
import time
for year in years:
    data = requests.get(standings_url2)
    soup = BeautifulSoup(data.text)
    standings_table = soup.select('table.stats_table')[0]

    links = [l.get("href") for l in standings_table.find_all('a')]
    links = [l for l in links if '/squads/' in l]
    team_urls = [f"https://fbref.com{l}" for l in links]
    
    previous_season = soup.select("a.prev")[0].get("href")
    standings_url2 = f"https://fbref.com{previous_season}"
    
    for team_url in team_urls:
        team_name = team_url.split("/")[-1].replace("-Stats", "").replace("-", " ")
        data = requests.get(team_url)
        matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
        soup = BeautifulSoup(data.text)
        links = [l.get("href") for l in soup.find_all('a')]
        links = [l for l in links if l and 'all_comps/shooting/' in l]
        data = requests.get(f"https://fbref.com{links[0]}")
        shooting = pd.read_html(data.text, match="Shooting")[0]
        shooting.columns = shooting.columns.droplevel()
        try:
            team_data = matches.merge(shooting[["Date", "Sh", "SoT", "Dist", "FK", "PK", "PKatt"]], on="Date")
        except ValueError:
            continue
        team_data = team_data[team_data["Comp"] == "Serie A"]
        
        team_data["Season"] = year
        team_data["Team"] = team_name
        all_matches.append(team_data)
        time.sleep(1)

/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykernel_9861/2166766338.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykernel_9861/2166766338.py:22: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  shooting = pd.read_html(data.text, match="Shooting")[0]
/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykernel_9861/2166766338.py:17: FutureWarning: Passing literal html to 'read_html' is deprecated and will be removed in a future version. To read from a literal string, wrap it in a 'StringIO' object.
  matches = pd.read_html(data.text, match="Scores & Fixtures")[0]
/var/folders/8h/b6sjd9l522v8k4p6t77m8fpm0000gn/T/ipykerne

ValueError: No tables found

In [43]:
all_matches

[         Date   Time     Comp        Round  Day Venue Result GF GA  \
 1  2024-08-18  18:30  Serie A  Matchweek 1  Sun  Away      L  0  3   
 2  2024-08-25  20:45  Serie A  Matchweek 2  Sun  Home      W  3  0   
 3  2024-08-31  20:45  Serie A  Matchweek 3  Sat  Home      W  2  1   
 4  2024-09-15  18:00  Serie A  Matchweek 4  Sun  Away      W  4  0   
 5  2024-09-21  18:00  Serie A  Matchweek 5  Sat  Away      D  0  0   
 7  2024-09-29  20:45  Serie A  Matchweek 6  Sun  Home      W  2  0   
 8  2024-10-04  18:30  Serie A  Matchweek 7  Fri  Home      W  3  1   
 
         Opponent  ...  Match Report  Notes  Sh  SoT  Dist   FK PK PKatt  \
 1  Hellas Verona  ...  Match Report    NaN  14    4  17.8  0.0  0     0   
 2        Bologna  ...  Match Report    NaN  16    5  17.5  0.0  0     0   
 3          Parma  ...  Match Report    NaN  29    7  17.5  1.0  0     0   
 4       Cagliari  ...  Match Report    NaN  13    5  16.0  0.0  0     0   
 5       Juventus  ...  Match Report    NaN   8   

In [44]:
match_df = pd.concat(all_matches)
match_df.columns = [c.lower() for c in match_df.columns]
match_df

,date,time,comp,round,day,venue,result,gf,ga,opponent,...,match report,notes,sh,sot,dist,fk,pk,pkatt,season,team
1,2024-08-18,18:30,Serie A,Matchweek 1,Sun,Away,L,0,3,Hellas Verona,...,Match Report,NaN,14,4,17.8,0.0,0,0,2022,Napoli
2,2024-08-25,20:45,Serie A,Matchweek 2,Sun,Home,W,3,0,Bologna,...,Match Report,NaN,16,5,17.5,0.0,0,0,2022,Napoli
3,2024-08-31,20:45,Serie A,Matchweek 3,Sat,Home,W,2,1,Parma,...,Match Report,NaN,29,7,17.5,1.0,0,0,2022,Napoli
4,2024-09-15,18:00,Serie A,Matchweek 4,Sun,Away,W,4,0,Cagliari,...,Match Report,NaN,13,5,16.0,0.0,0,0,2022,Napoli
5,2024-09-21,18:00,Serie A,Matchweek 5,Sat,Away,D,0,0,Juventus,...,Match Report,NaN,8,1,23.4,0.0,0,0,2022,Napoli
7,2024-09-29,20:45,Serie A,Matchweek 6,Sun,Home,W,2,0,Monza,...,Match Report,NaN,11,2,13.8,0.0,0,0,2022,Napoli
8,2024-10-04,18:30,Serie A,Matchweek 7,Fri,Home,W,3,1,Como,...,Match Report,NaN,8,5,18.1,0.0,1,1,2022,Napoli
0,2024-08-17,18:30,Serie A,Matchweek 1,Sat,Away,D,2.0,2.0,Genoa,...,Match Report,NaN,17,7,13.7,1.0,0,0,2022,Internazionale
1,2024-08-24,20:45,Serie A,Matchweek 2,Sat,Home,W,2.0,0.0,Lecce,...,Match Report,NaN,10,2,15.2,1.0,1,1,2022,Internazionale
2,2024-08-30,20:45,Serie A,Matchweek 3,Fri,Home,W,4.0,0.0,Atalanta,...,Match Report,NaN,13,4,17.9,0.0,0,0,2022,Internazionale


In [45]:
match_df.to_csv("matches.csv")